# **Yolov5 Object Detection - NYC**

#### **Motivation:** Project that utilizes YoloV5's pre-trained CNN to detect objects within a video in NYC (Timelapse footage of people walking inside Grand Central Station in New York.)

**Video Source:** https://coverr.co/videos/timelapse-of-grand-central-station-t0ug9xvkec

## **Documentation:**

### Pre-Req Vocab:

* **write (within the context of OpenCV's VideoWriter module)** = To output or save processed video frames into a video file
    * Writing individual frames to a video file
    * Each frame in most cases is an image within a video that has been processed (*videos are made up of a multitude of images*)
* **fourcc** = Codec information obtained from videowriter_fourcc
* **fps** = frames per second
* **frame-size** = Size of video frames (width, height)


### Yolov5 Model Attributes:

* **model.Stride** = Stride of Model, downsampling factor of the pre-trained model, affecting the resolution of the feature maps
* **model.Names** = List of class names that the model can detect (essentially labels (thing we are predicting aka y-hat))
* **model.pt** = model file formatting, .pt does not have a documented meaning but uses pickle to serialize model weights/params

### VideoWriter Object:
* **Motivation:** Helps us write the processed frames to an output video file
* **Components:**
    * **Codec:** = Software/Hardware used to compress and de-compress digital videos
    * **Codec Used:** fourcc (Four Character code of the codec used to compress the video frames (.mp4v stands for MPEG-4 video codec)
    * **Retrieval of Frame Dimensions:**
        * Code opens up a video file and reads its properties
        * Retrieves the width and height of the frames using CV2 Library (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT)
        * Releases the video capture object as it is no longer needed for reading properties
    * **Creation of VideoWriter Object:**
        * Initializing the VideoWriter Object
        * 30.0 = Frame Rate of Video
        * Size of Video Frames = (frame_width, frame_height)

### Processing Each Frame:
* **Motivation:** 
    * Processing each frame of the input video passed in
    * Run Object Detection (using Yolov5)
    * Write Processed Frames to the output video
* **Components:**
    * For loop that Loops through the processed video frames
    * Converting video frames to tensors (a commonly used numerical data structure that represents data as embeddings)
        * Converts frame from NumPy array to PyTorch tensor and moves it to the GPU for processing
        * Normalizes the pixel values to the range [0,1] through the division of 255
        * Adds a batch dimension if the tensor has only 3 dimensions
    * **Running Inference:**
        * Calling and running the pre-trained Yolov5 model on the video frames to get predictions
    * **Applying Non-Max Supression:**
        * Applies NMS (Non-Max Supression) to filter out overlapping bounding boxes, keeping the most confident ones
    * **Processing Detections:**
        * for loop that iterates through each detection processed, creating a copy of the frame for annotation
        * scaling the bounding boxes back to the original frame size
        * annotating the frame with bounding boxes and labels
    * resizing and writing frame to video
        * resizes frame to ensure it matches the dimensions specified in the video-writer object
        * writes the annotated frame to the output video
        
### Releasing Resources:
* **Motivation:**
    * Releasing the VideoWriter resources and displaying the processed video within the notebook

### Cloning & Setting up Enviromental Variables & Dependencies pre-code

In [ ]:
#Cloning the yolov5 repo into the environment
!git clone https://github.com/ultralytics/yolov5
%cd yolov5

# Installing Necessary Dependencies within the yolov5 .txt dependencies list
!pip install -r requirements.txt

In [ ]:
import sys
sys.path.append('/kaggle/working/yolov5')
import os
print(os.listdir('utils'))

# Code Post-Clone

In [3]:
### Main Dependencies
import cv2
import torch
from models.common import DetectMultiBackend
from utils.dataloaders import LoadImages
from utils.general import non_max_suppression, scale_boxes
from utils.plots import Annotator, colors
from utils.torch_utils import select_device

# Processing devices selected for processing, I'm on Kaggle, using Tesla T4 X2 GPU's, hence '(0,1)' since I'm using 2 GPU's
device = select_device('0,1')

# Loading in the pre-trained YOLOv5 model
model = DetectMultiBackend('yolov5s.pt', device=device) #yolov5.pt links to the yolov5 pre-trained model weights (params)
stride, names, pt = model.stride, model.names, model.pt

# Opening the video file
video_path = '/kaggle/input/nyc-video/coverr-timelapse-of-grand-central-station-7681-1080p.mp4'  # Replace with your video path
dataset = LoadImages(video_path, img_size=640, stride=stride)

# Defining the fourcc codec and creating the VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# Retrieving the frame width and height from the video capture
vid_cap = cv2.VideoCapture(video_path)
frame_width = int(vid_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(vid_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
vid_cap.release()

out = cv2.VideoWriter('/kaggle/working/output_video.mp4', fourcc, 30.0, (frame_width, frame_height))

# Looping through the video frames
# path = path to current video frame
# img = the current video frame/img(image) as a numpy array, preprocessed for the yolov5 model
# im0s = the original frame as a numpy array, planned to be used for displaying the results
# vid_cap = video capture object for accessing video properties
# s = string used for displaying processing information
for path, img, im0s, vid_cap, s in dataset:
    img = torch.from_numpy(img).to(device) #converting the video frame from a numpy array to a PyTorch tensor, moving the tensor to the specified device (GPU)
    img = img.float() / 255.0  # Normalizing the image pixel values to the range of [0, 1], converting the tensor to floating-point numbers
    if len(img.shape) == 3:
        img = img[None]  # Adding batch dimension

    # Inference
    pred = model(img, augment=False, visualize=False) #disabling augmentations and visualization

    #Applying NMS (non-max suppression) = NMS is a function that filters out overlapping bounding boxes, keeping only the most confident ones
    #pred = raw predictions from the model
        #0.25 = Confidence Threshold
        #0.45 = IoU (Intersection over Union threshold for NMS, overlapping boxes with IoU above this value are suppressed)
        #None = no class-specific NMS threshold used, False = Whether to apply NMS for multi-class
        #max_det=1000, max number of detections to keep
    pred = non_max_suppression(pred, 0.25, 0.45, None, False, max_det=1000)

    # Processing frame detections
    for i, det in enumerate(pred):  #iterating over a list of predictions (one for each image in the batch)
        im0 = im0s.copy() #copying the original frame to annotate it with decision results
        annotator = Annotator(im0, line_width=2, example=str(names)) #im0 = frame to annotate, line_width=2 is the width of the bounding box lines, example=str(names) provides a string example of class names
        if len(det): #checking for detections, scale_boxes = adjusting bounding box coords, img.shape[2:] are the dimensions of the model input, det[:, :4] are the bounding box coordinates in the model output, im0.shape retrieves the dimensions of the original frame, round(), rounds the coordinates to integer values for drawing
            det[:, :4] = scale_boxes(img.shape[2:], det[:, :4], im0.shape).round()

            # Writing results
            for *xyxy, conf, cls in reversed(det):
                label = f'{names[int(cls)]} {conf:.2f}'
                annotator.box_label(xyxy, label, color=colors(cls))

        # Ensuring frame size matches
        frame = cv2.resize(im0, (frame_width, frame_height))

        # Writing frame to video
        out.write(frame)

# Releasing everything once the processing is complete
out.release()

YOLOv5 🚀 v7.0-331-gab364c98 Python-3.10.13 torch-2.1.2 CUDA:0 (Tesla T4, 15102MiB)
                                                        CUDA:1 (Tesla T4, 15102MiB)

100%|██████████| 14.1M/14.1M [00:00<00:00, 127MB/s]

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
WARNING ⚠️ NMS time limit 0.550s exceeded
